# GPT Prediction Visualization

This notebook visualizes what GPT predicts at each position, not just the final next token.

For a given prompt, we show:
- What the model has seen so far (context)
- Top predictions for the next token at each position
- Probabilities and confidence
- How predictions change as context grows

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import jax
import jax.numpy as jnp
import numpy as np
import tiktoken
from IPython.display import display, HTML
import pandas as pd

from janogpt import Config, GPT
from pretrained.huggingface.loader import load_hf_gpt2_weights

print(f"JAX devices: {jax.devices()}")
print(f"JAX version: {jax.__version__}")

## Load Pretrained GPT-2

In [ ]:
# Create GPT-2 config (matches HuggingFace GPT-2)
config = Config(
    num_blocks=12,
    emb_dim=768,
    num_heads=12,
    seq_len=1024,
    voc_size=50257,  # HuggingFace vocab size
    dropout_prob=0.0,  # No dropout for inference
)

print("Creating model...")
model = GPT(config)

print("Loading pretrained weights from HuggingFace...")
params = load_hf_gpt2_weights(model, config, model_name="gpt2")

# Count parameters
param_count = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"✓ Model loaded: {param_count / 1e6:.1f}M parameters")

# Load tokenizer
print("Loading tokenizer...")
enc = tiktoken.get_encoding("gpt2")
print(f"✓ Tokenizer loaded: {enc.n_vocab} tokens")

## Define Visualization Function

In [ ]:
def visualize_predictions(prompt, top_k=5):
    """
    Visualize what GPT predicts at each position.
    
    Args:
        prompt: Input text
        top_k: Number of top predictions to show
    """
    # Tokenize
    tokens = enc.encode(prompt)
    input_ids = jnp.array([tokens], dtype=jnp.int32)  # (1, T)
    
    print(f"Prompt: '{prompt}'")
    print(f"Tokens: {len(tokens)}")
    print()
    
    # Forward pass
    logits = model.apply(
        {'params': params},
        input_ids,
        inference=True
    )  # (1, T, vocab_size)
    
    # Get probabilities
    probs = jax.nn.softmax(logits[0], axis=-1)  # (T, vocab_size)
    
    # Create visualization table
    rows = []
    
    for pos in range(len(tokens)):
        # Context so far
        context_tokens = tokens[:pos+1]
        context_text = enc.decode(context_tokens)
        
        # Current token
        current_token = enc.decode([tokens[pos]])
        
        # Top predictions for NEXT token
        pos_probs = probs[pos]  # Predictions after seeing tokens[0:pos+1]
        top_indices = jnp.argsort(pos_probs)[-top_k:][::-1]
        top_probs = pos_probs[top_indices]
        
        # Format top predictions
        predictions = []
        for idx, prob in zip(top_indices, top_probs):
            token_text = enc.decode([int(idx)])
            predictions.append(f"{token_text!r} ({float(prob)*100:.1f}%)")
        
        # Actual next token (if available)
        if pos < len(tokens) - 1:
            actual_next = enc.decode([tokens[pos + 1]])
            actual_prob = float(pos_probs[tokens[pos + 1]]) * 100
            
            # Check if actual next token is in top-k
            is_top_k = tokens[pos + 1] in top_indices
            actual_str = f"{actual_next!r} ({actual_prob:.1f}%)" + (" ✓" if is_top_k else " ✗")
        else:
            actual_str = "(end)"
        
        rows.append({
            'Position': pos,
            'Context': context_text,
            'Current Token': current_token,
            f'Top {top_k} Predictions': ', '.join(predictions),
            'Actual Next': actual_str,
        })
    
    # Create DataFrame
    df = pd.DataFrame(rows)
    
    # Style function for better readability
    def highlight_correct(val):
        if '✓' in str(val):
            return 'background-color: lightgreen'
        elif '✗' in str(val):
            return 'background-color: lightcoral'
        return ''
    
    styled_df = df.style.applymap(highlight_correct, subset=['Actual Next'])
    
    return styled_df

## Example 1: Simple Sentence

In [ ]:
visualize_predictions("The cat sat on the", top_k=5)

## Example 2: Factual Knowledge

In [ ]:
visualize_predictions("The capital of France is", top_k=5)

## Example 3: Math/Reasoning

In [ ]:
visualize_predictions("2 + 2 equals", top_k=5)

## Example 4: Your Custom Prompt

Try your own prompt here!

In [ ]:
# Change this to your prompt
custom_prompt = "Once upon a time"

visualize_predictions(custom_prompt, top_k=5)

## Advanced: Heatmap Visualization

Visualize prediction probabilities as a heatmap across all positions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_prediction_heatmap(prompt, num_tokens=20):
    """
    Show heatmap of top token probabilities at each position.
    
    Args:
        prompt: Input text
        num_tokens: Number of top tokens to show in heatmap
    """
    # Tokenize
    tokens = enc.encode(prompt)
    input_ids = jnp.array([tokens], dtype=jnp.int32)
    
    # Forward pass
    logits = model.apply({'params': params}, input_ids, inference=True)
    probs = jax.nn.softmax(logits[0], axis=-1)  # (T, vocab_size)
    
    # Get top tokens across all positions
    all_top_indices = set()
    for pos in range(len(tokens)):
        top_k_pos = jnp.argsort(probs[pos])[-num_tokens:]
        all_top_indices.update([int(idx) for idx in top_k_pos])
    
    # Create matrix: (positions, top_tokens)
    top_token_list = sorted(list(all_top_indices))[:num_tokens]
    prob_matrix = np.zeros((len(tokens), len(top_token_list)))
    
    for pos in range(len(tokens)):
        for i, token_idx in enumerate(top_token_list):
            prob_matrix[pos, i] = float(probs[pos, token_idx])
    
    # Token labels
    token_labels = [enc.decode([t])[:10] for t in tokens]  # Truncate long tokens
    top_token_labels = [enc.decode([t])[:10] for t in top_token_list]
    
    # Plot
    plt.figure(figsize=(14, max(8, len(tokens) * 0.5)))
    sns.heatmap(
        prob_matrix,
        xticklabels=top_token_labels,
        yticklabels=token_labels,
        cmap='YlOrRd',
        cbar_kws={'label': 'Probability'},
        annot=True,
        fmt='.2f',
        linewidths=0.5
    )
    plt.xlabel('Predicted Token')
    plt.ylabel('Position (Current Token)')
    plt.title(f'Prediction Probabilities: "{prompt}"')
    plt.tight_layout()
    plt.show()
    
    print(f"\nInterpretation:")
    print(f"- Each row shows predictions AFTER seeing that token")
    print(f"- Brighter colors = higher probability")
    print(f"- Shows how predictions change as context grows")

In [ ]:
visualize_prediction_heatmap("The cat sat on the", num_tokens=10)

## Confidence Analysis

Analyze how confident the model is at each position.

In [ ]:
def analyze_confidence(prompt):
    """
    Analyze model confidence at each position.
    """
    # Tokenize
    tokens = enc.encode(prompt)
    input_ids = jnp.array([tokens], dtype=jnp.int32)
    
    # Forward pass
    logits = model.apply({'params': params}, input_ids, inference=True)
    probs = jax.nn.softmax(logits[0], axis=-1)  # (T, vocab_size)
    
    # Calculate metrics
    max_probs = jnp.max(probs, axis=-1)  # Confidence (max probability)
    entropies = -jnp.sum(probs * jnp.log(probs + 1e-10), axis=-1)  # Uncertainty
    
    # If we know the actual next tokens, calculate their probabilities
    actual_probs = []
    for pos in range(len(tokens) - 1):
        next_token = tokens[pos + 1]
        prob = float(probs[pos, next_token])
        actual_probs.append(prob)
    actual_probs.append(None)  # No next token for last position
    
    # Create DataFrame
    token_strs = [enc.decode([t]) for t in tokens]
    df = pd.DataFrame({
        'Position': range(len(tokens)),
        'Token': token_strs,
        'Max Prob (Confidence)': [f"{float(p)*100:.1f}%" for p in max_probs],
        'Entropy (Uncertainty)': [f"{float(e):.2f}" for e in entropies],
        'Actual Next Token Prob': [f"{p*100:.1f}%" if p is not None else "N/A" for p in actual_probs],
    })
    
    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # Confidence over positions
    axes[0].plot(range(len(tokens)), [float(p) for p in max_probs], marker='o', linewidth=2)
    axes[0].set_xlabel('Position')
    axes[0].set_ylabel('Confidence (Max Probability)')
    axes[0].set_title('Model Confidence Across Positions')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0, 1])
    
    # Entropy over positions
    axes[1].plot(range(len(tokens)), [float(e) for e in entropies], marker='s', linewidth=2, color='orange')
    axes[1].set_xlabel('Position')
    axes[1].set_ylabel('Entropy (Uncertainty)')
    axes[1].set_title('Prediction Uncertainty Across Positions')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return df

In [ ]:
confidence_df = analyze_confidence("The capital of France is Paris")
display(confidence_df)

## Key Insights

From these visualizations, you can observe:

1. **Context Matters**: Predictions change dramatically as more context is added
2. **Confidence Varies**: Model is more confident on some tokens (e.g., "Paris" after "capital of France is")
3. **Top-k Coverage**: Check how often the actual next token appears in top-k predictions (✓ vs ✗)
4. **Uncertainty**: High entropy = model is uncertain, low entropy = confident
5. **Common Patterns**: Certain word combinations have very predictable next tokens

This helps understand:
- Why sampling with temperature works (flattens the distribution)
- Why top-k sampling helps (focuses on likely tokens)
- How the model "thinks" about next-token prediction